# Case 4 — Latent dimensionality: MLP needs capacity, LSTM doesn't

**Reproduces:** Fig 4.12

Contextual anomalies, window mode, no cyclic features (so any seasonal structure has to come from the latent code alone). MLP-VAE has no other way to represent 'what time of year is this', so it should need a much larger bottleneck (Z=48) than at Z=3. LSTM-VAE's recurrence already carries most of the trajectory, so its latent code mainly needs to encode the residual — expect it to be far less sensitive to Z.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the small synthetic ERA5-shaped dataset shipped with the repo (`scripts/generate_mini_era5.py`) — no data download, no license issues. Numbers will differ from the thesis's real-ERA5 figures (much smaller warmup/test period, noisier), but the *qualitative* effect described above should still show up.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# TODO: update this URL once the repo is pushed to GitHub
REPO_URL = "https://github.com/<your-username>/streaming-vae-anomaly-detection.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Generates data/era5/*.csv — fully synthetic, ERA5-shaped, no download needed
!python scripts/generate_mini_era5.py

## Run the suite

`notebooks/cases/case04_latent_dimension_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially on the one Colab GPU; each is small (mini dataset), so the whole case should finish in a few minutes.

In [ ]:
!python run_regression.py notebooks/cases/case04_latent_dimension_suite.yaml \
    --session runs/regression/case04_latent_dimension --gpus 0

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case04_latent_dimension

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case04_latent_dimension/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Display the key comparison plot(s) inline
import glob
from IPython.display import Image, display

print("F1 vs latent_dim, split by architecture:")
for p in sorted(glob.glob("runs/regression/case04_latent_dimension/cross_compare/contextual/section_lines_latent_dim.png")):
    display(Image(filename=p))